# 09 · Selección de variables

Este notebook decide qué variables entran en el modelo, agrupa las máquinas con muy pocas fases y genera el dataset de
modelado. La codificación de las variables categóricas depende de cada modelo y se hace dentro de cada notebook de
modelado (one-hot en Ridge, ordinal en Random Forest y tratamiento nativo en LightGBM).

- **Entrada:** `data/processed/df_entrenamiento.parquet`
- **Salida:** `data/processed/df_features.parquet`

In [27]:
import sys
from pathlib import Path

# Raíz del repositorio (la carpeta que contiene src/), para poder importar src
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import pandas as pd

from src.rutas import PROCESSED

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

## 1. Criterio: solo información disponible al planificar

El problema es de idoneidad fase-máquina: el modelo predice el ratio a partir de características disponibles antes de ejecutar la fase. Todo predictor tiene que ser información que existe en el momento de decidir a qué máquina se asigna la fase. Usar algo que solo se conoce después de ejecutarla es una fuga de información (*data leakage*): el modelo parecería funcionar muy bien en las pruebas, pero fallaría al usarse de verdad.
     
El caso más claro es el tiempo real. El ratio es tiempo real / tiempo teórico, y el tiempo real se conoce después de ejecutar la fase, así que `tiempo_real_efectivo` no puede ser predictor: sería usar parte de la respuesta para predecir la respuesta. Por la misma razón, en el notebook 08 ya se eliminaron `dedicacion_media`, `n_imputaciones` y los minutos reales (`MinutosReal_*`): todos se conocen solo tras ejecutar la fase.
     
    
## 2. Variables elegidas y descartadas

| Columna | Uso | Motivo |
|---|---|---|
| `smFaseNme` → `TipoFase` | Predictor | Tipo de fase; se conoce al planificar |
| `MaquinaDominante` → `CodigoMaquina` | Predictor | Máquina asignada: es la decisión que se evalúa |
| `CodigoGrupo` → `Familia Maquinas` | Predictor | Familia de la máquina asignada a la fase|
| `CodProductLine` → `Línea Producto` | Predictor | Línea de producto del artículo |
| `CodProductFamily` → `Família Producto` | Predictor | Familia de producto del artículo |
| `tiempo_previsto` → `TiempoPrevisto` | Predictor | Tiempo teórico del ERP |
| `Quantity` | Predictor | Cantidad a fabricar |
| `ratio` | Variable objetivo | Tiempo real efectivo / tiempo previsto |
| `fecha_fase` | Solo para la partición temporal | No es predictor |
| `IDMOTask`, `smOrdenId` | Claves para la partición | Identifican la fase y la orden |
| `tiempo_real_efectivo` | Descartada | Fuga de información (forma parte del ratio) |
| `IDUsualTask` | Descartada | Redundante |
| `smMaterialId` | Descartada | Cardinalidad muy alta (3.939 artículos distintos) |
| `ProductEntry`, `MaterialExit` | Descartadas | No es relevante para la desviación de tiempos |
| `smOrdenNme`, `Description`, `NombreMaquina`, `NombreGrupo`, `Descripción Artículo`, `Nombre Línea`, `Nombre Família` | Descartadas | Textos descriptivos que repiten la información de los códigos |

In [28]:
df_entrenamiento = pd.read_parquet(PROCESSED / "df_entrenamiento.parquet")
print(df_entrenamiento.shape)

(14689, 23)


## 3. Máquinas con pocas fases

Número de fases por máquina:

In [29]:
fases_por_maquina = df_entrenamiento["MaquinaDominante"].value_counts()
print(f"Máquinas: {len(fases_por_maquina)}\n")
print(fases_por_maquina.to_string())

Máquinas: 28

MaquinaDominante
TN09    3224
TP02    3188
CM06    1286
CM05     908
TN04     864
CM14     708
CM16     600
TN08     554
TN10     504
CM12     439
TN05     423
TP01     414
CM08     318
CM01     211
PH04     175
FL01     168
CM10     154
PH03     139
TN07     128
CM07     117
CM15      86
CM19      39
CM17      21
CM04      17
CM13       1
CM11       1
CM18       1
MA02       1


La inspección del número de fases por máquina muestra un salto claro entre las máquinas con actividad habitual, que acumulan decenas o cientos de fases, y un pequeño grupo con apenas registros. Este hueco permite fijar un umbral de corte en 30 fases. Las máquinas por debajo del umbral se agrupan en una categoría "Otras" de su familia (`Otras_<familia>`), conservando su familia de máquina como variable, de forma que el modelo pueda seguir aprovechando esa información aunque no disponga de un patrón específico por máquina. Es la mejor opción para no
descartar estos datos. En un uso práctico futuro, a medida que estas máquinas registren más imputaciones, pasarán a tener su propia categoría.

> Esta cambio no puede hacerse para el globlal de los datos, se hara para los datos de entrenamiento SOLO, en el notebook `10_split_temporal`.  

Entonces a modo exploratorio:

In [30]:
MIN_FASES_MAQUINA = 30

maquinas_raras = fases_por_maquina[fases_por_maquina < MIN_FASES_MAQUINA].index
print(f"Máquinas a agrupadar en 'Otras' ({len(maquinas_raras)}): {list(maquinas_raras)}")

Máquinas a agrupadar en 'Otras' (6): ['CM17', 'CM04', 'CM13', 'CM11', 'CM18', 'MA02']


La agrupación se aplica en el notebook 10, contando las fases solo en el train, para no usar información del test

In [31]:
df_features = df_entrenamiento.drop(columns=[
    "MaterialExit", "ProductEntry", "smMaterialId", "IDUsualTask", "tiempo_real_efectivo",
    "smOrdenNme", "Description", "NombreMaquina", "NombreGrupo",
    "Descripción Artículo", "Nombre Línea", "Nombre Família",
]).rename(columns={
    "smFaseNme": "TipoFase",
    "MaquinaDominante": "CodigoMaquina",
    "CodigoGrupo": "Familia Maquinas",
    "CodProductLine": "Línea Producto",
    "CodProductFamily": "Família Producto",
    "tiempo_previsto": "TiempoPrevisto",
})

maquinas_familia = df_features.groupby(["Familia Maquinas"])["CodigoMaquina"].nunique()

display(maquinas_familia)
print(f'Total de máqinas: {maquinas_familia.sum()}')

print(df_features.shape)
df_features.head(3)

Familia Maquinas
C5        4
CH        3
CON       2
CV        4
CV-ALU    3
FLEX      4
MTP       2
TCN       4
TPU       2
Name: CodigoMaquina, dtype: int64

Total de máqinas: 28
(14689, 11)


,fecha_fase,IDMOTask,smOrdenId,TipoFase,CodigoMaquina,Familia Maquinas,Línea Producto,Família Producto,Quantity,TiempoPrevisto,ratio
0,2025-10-02 03:12:05.010,ee76d685-a4cc-47d4-b418-6e33d2ae2921,OF2509545,TPP,TN09,TPU,P,PE,3.0,21.0,1.988889
1,2025-10-21 12:26:43.433,3c6cccb8-2326-4d66-9714-b09ac436be82,OF2510311,TMO,TN10,TCN,E,ELE,5.0,50.0,0.826333
2,2026-01-23 09:40:53.233,91871d20-4755-4ab9-a993-2cb6d76f1aed,OF2600009,TPG-CNY,TN05,TCN,B,BR,1.0,55.0,0.224243


Resumen de las variables categóricas:

In [32]:
cols_texto = df_features.select_dtypes(include=["object", "category"]).columns
pd.DataFrame({
    "valores distintos": df_features[cols_texto].nunique(),
    "% distintos": (df_features[cols_texto].nunique() / len(df_features) * 100).round(2),
    "nulos": df_features[cols_texto].isna().sum(),
})

,valores distintos,% distintos,nulos
IDMOTask,14689,100.00,0
smOrdenId,6501,44.26,0
TipoFase,38,0.26,0
CodigoMaquina,28,0.19,0
Familia Maquinas,9,0.06,0
Línea Producto,17,0.12,95
Família Producto,44,0.30,109


`Línea Producto` y `Família Producto` tienen valores nulos (fases sin línea o familia de producto). Cada modelo los
etiqueta como "Desconocida" (notebooks 12 a 14).

## 5. Guardado

In [33]:
df_features.to_parquet(PROCESSED / "df_features.parquet")